### Optimize Prompts:

MLflow's prompt optimization lets you systematically enhance your AI applications with minimal code changes. 

MLflow supports multiple optimization algorithms to improve your prompts:

- GEPA (GepaPromptOptimizer): Iteratively refines prompts using LLM-driven reflection and automated feedback, achieving systematic improvements through trial-and-error learning.

- Metaprompting (MetaPromptOptimizer): Restructures prompts to be more systematic and effective, working in both zero-shot mode (without training data) and few-shot mode (learning from your examples).

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

from langchain.chat_models import init_chat_model


# Now you can access your environment variables using os.environ
os.environ['OPENAI_API_KEY'] = os.environ.get("OPENAI_API_KEY")

llm = init_chat_model("gpt-5-mini-2025-08-07") 

In [2]:
import mlflow

# Calling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

# Optional: Set a tracking URI and an experiment
mlflow.set_experiment("prompt_optimization_experiment")
mlflow.set_tracking_uri("http://localhost:5000")

/Users/aritrasen/Documents/code/github/mlflow_tutorial/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/13 17:17:27 INFO mlflow.tracking.fluent: Experiment with name 'prompt_optimization_experiment' does not exist. Creating a new experiment.


### Register base prompt for optimization

In [3]:
import mlflow
import openai
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness

# Register initial prompt for classifying medical paper sections
prompt = mlflow.genai.register_prompt(
    name="medical_section_classifier",
    template="Classify this medical research paper sentence into one of these sections: " \
    "CONCLUSIONS, RESULTS, METHODS, OBJECTIVE, BACKGROUND.\n\nSentence: {{sentence}}",
)

2026/06/13 17:19:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: medical_section_classifier, version 1


### Predict Function

In [12]:
def predict_fn(sentence: str) -> str:
    prompt = mlflow.genai.load_prompt("prompts:/medical_section_classifier/1")
    response = llm.invoke(
        [
            {
                "role": "user",
                "content": prompt.format(sentence=sentence, num_sentences=1),
            }
        ],
    )
    return response.content

### Training data with medical paper sentences and ground truth labels

In [7]:

raw_data = [
    ("The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .", "BACKGROUND"),
    ("This paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .", "BACKGROUND"),
    ("This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .", "METHODS"),
    ("The intervention group will participate in the online group program ` Positive Outlook ' .", "METHODS"),
    ("Results of the Positive Outlook study will provide information regarding the effectiveness of online group programs improving health related outcomes for men living with HIV .", "CONCLUSIONS"),
    ("The aim of this study was to evaluate the efficacy , safety and complications of orbital steroid injection versus oral steroid therapy in the management of thyroid-related ophthalmopathy .", "OBJECTIVE"),
    ("A total of 29 patients suffering from thyroid ophthalmopathy were included in this study .", "METHODS"),
    ("Patients were randomized into two groups : group I included 15 patients treated with oral prednisolone and group II included 14 patients treated with peribulbar triamcinolone orbital injection .", "METHODS"),
    ("Both groups showed improvement in symptoms and in clinical evidence of inflammation with improvement of eye movement and proptosis in most cases .", "RESULTS"),
    ("Mean exophthalmometry value before treatment was 22.6 1.98 mm that decreased to 18.6 0.996 mm in group I , compared with 23 1.86 mm that decreased to 19.08 1.16 mm in group II .", "RESULTS"),
    ("There was no change in the best-corrected visual acuity in both groups .", "RESULTS"),
    ("There was an increase in body weight , blood sugar , blood pressure and gastritis in group I in 66.7 % , 33.3 % , 50 % and 75 % , respectively , compared with 0 % , 0 % , 8.3 % and 8.3 % in group II .", "RESULTS"),
    ("Orbital steroid injection for thyroid-related ophthalmopathy is effective and safe .", "CONCLUSIONS"),
    ("It eliminates the adverse reactions associated with oral corticosteroid use .", "CONCLUSIONS"),
    ("The aim of this prospective randomized study was to examine whether active counseling and more liberal oral fluid intake decrease postoperative pain , nausea and vomiting in pediatric ambulatory tonsillectomy .", "OBJECTIVE"),
]

# Format dataset for optimization

dataset = [
    {
        "inputs": {"sentence": sentence},
        "expectations": {"expected_response": label},
    }
    for sentence, label in raw_data
]

In [8]:
dataset[:3]  # Display the first 3 entries of the dataset

[{'inputs': {'sentence': 'The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition , including making physical , emotional and social adjustments .'},
  'expectations': {'expected_response': 'BACKGROUND'}},
 {'inputs': {'sentence': 'This paper describes the design and evaluation of Positive Outlook , an online program aiming to enhance the self-management skills of gay men living with HIV .'},
  'expectations': {'expected_response': 'BACKGROUND'}},
 {'inputs': {'sentence': 'This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .'},
  'expectations': {'expected_response': 'METHODS'}}]

In [17]:
! uv add 'gepa>=0.0.26' litellm

Resolved 208 packages in 3.25s                                       
⠹ Preparing packages... (0/11)                                                  
⠸ Preparing packages... (0/11)-------------     0 B/41.33 KiB           
⠸ Preparing packages... (0/11)-------------     0 B/41.33 KiB           
⠸ Preparing packages... (0/11)------------- 16.00 KiB/41.33 KiB         
⠸ Preparing packages... (0/11)--------- 32.00 KiB/41.33 KiB         
⠸ Preparing packages... (0/11)--------- 41.33 KiB/41.33 KiB         
⠸ Preparing packages... (0/11)--------- 41.33 KiB/41.33 KiB         
importlib-metadata   ------------------------------     0 B/26.58 KiB
⠸ Preparing packages... (0/11)--------- 41.33 KiB/41.33 KiB         
⠸ Preparing packages... (0/11)-------------     0 B/26.58 KiB           
⠸ Preparing packages... (0/11)-------------     0 B/26.58 KiB           
importlib-metadata   ------------------------------     0 B/26.58 KiB
⠸ Preparing packages... (0/11)-------------     0 B/89.54 KiB       

### Optimize the prompt

GEPA (Genetic-Pareto) optimization algorithm to optimize prompts:

GEPA uses iterative mutation, reflection, and Pareto-aware candidate selection to improve text components like prompts. It leverages large language models to reflect on system behavior and propose improvements.

In [22]:
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="openai:/gpt-5", max_metric_calls=50,display_progress_bar=True),
    scorers=[Correctness(model="openai:/gpt-5-mini")],
)

2026/06/13 17:40:59 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
/Users/aritrasen/Documents/code/github/mlflow_tutorial/.venv/lib/python3.13/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
GEPA Optimization:  30%|███       | 15/50 [00:12<00:28,  1.21rollouts/s]

Iteration 0: Base program full valset score: 0.9333333333333333 over 15 / 15 examples
Iteration 1: Selected program 0 score: 0.9333333333333333
Iteration 1: Proposed new text for medical_section_classifier: You will be given a single sentence from a medical research paper. Classify it into exactly one of these sections:
- CONCLUSIONS
- RESULTS
- METHODS
- OBJECTIVE
- BACKGROUND

Output rules:
- Return only the label in ALL CAPS with no extra text, punctuation, or explanation.

How to decide (use these cues and prioritize in this order when clear signals exist):

1) RESULTS
- Reports outcomes/findings, often past tense.
- Frequently includes numbers, percentages, p-values, confidence intervals, effect sizes, or comparative statements (including “no change”).
- Examples of cues: “There was an increase…”, “significantly higher/lower…”, “OR, RR, HR…”, “(p < …)”, “mean ± SD”, “compared with…”.
- Note: Absence of numbers doesn’t exclude RESULTS (e.g., “There was no change in visual acuity in

GEPA Optimization:  42%|████▏     | 21/50 [01:08<01:53,  3.90s/rollouts]

Iteration 1: New subsample score 0.0 is not better than old score 2.0, skipping
Iteration 2: Selected program 0 score: 0.9333333333333333


GEPA Optimization:  48%|████▊     | 24/50 [01:14<01:31,  3.51s/rollouts]

...

Iteration 8: All subsample scores perfect. Skipping.
Iteration 8: Reflective mutation did not propose a new candidate
Iteration 9: Selected program 0 score: 0.9333333333333333
Iteration 9: Proposed new text for medical_section_classifier: You are classifying a single sentence from a medical research paper into one of exactly five section labels:
- CONCLUSIONS
- RESULTS
- METHODS
- OBJECTIVE
- BACKGROUND

Output requirements:
- Output only one of the five labels above, in ALL CAPS, with no extra words, punctuation, or explanation.

General approach:
- Use linguistic cues in the sentence to infer its section, since you may not have the full abstract.
- Prefer the most specific cue present (e.g., explicit aim phrasing → OBJECTIVE; reporting data/significance → RESULTS).
- When in doubt between RESULTS and CONCLUSIONS, ask: is the sentence reporting what was found (RESULTS) or interpreting/recommending/claiming implications (CONCLUSIONS)?

Section cues and decision rules:

1) OBJECTIVE
- P

GEPA Optimization:  96%|█████████▌| 48/50 [03:15<00:14,  7.13s/rollouts]

Iteration 9: New subsample score 0.0 is not better than old score 2.0, skipping
Iteration 10: Selected program 0 score: 0.9333333333333333
Iteration 10: Proposed new text for medical_section_classifier: You are given a single sentence from a medical research paper. Classify it into exactly one of these sections:
- CONCLUSIONS
- RESULTS
- METHODS
- OBJECTIVE
- BACKGROUND

Return only the section label in ALL CAPS with no extra text, punctuation, or explanations.

Decision guide (use these cues and tie-breakers):

1) CONCLUSIONS
- Interpretive takeaways, implications, recommendations, overall statements of effectiveness/safety.
- Often includes phrases like: “In conclusion,” “We conclude,” “These findings suggest/indicate,” “is effective and safe,” “supports,” “warrants.”
- Typically no detailed numbers.
- Example cue: “Orbital steroid injection … is effective and safe.”

2) RESULTS
- Empirical findings and outcome values: numbers, changes, comparisons, p-values, CIs, effect sizes.
- Phr

GEPA Optimization:  96%|█████████▌| 48/50 [04:07<00:10,  5.15s/rollouts]
2026/06/13 17:45:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: medical_section_classifier, version 3


Iteration 10: New subsample score 2.0 is not better than old score 2.0, skipping
🏃 View run charming-ape-822 at: http://localhost:5000/#/experiments/6/runs/40afd1b5cdba431eaa914965e627f89f
🧪 View experiment at: http://localhost:5000/#/experiments/6


In [23]:
# Use the optimized prompt
optimized_prompt = result.optimized_prompts[0]
print(f"Optimized template: {optimized_prompt.template}")   

Optimized template: Classify this medical research paper sentence into one of these sections: CONCLUSIONS, RESULTS, METHODS, OBJECTIVE, BACKGROUND.

Sentence: {{sentence}}
